<a href="https://colab.research.google.com/github/damianwgriggs/Quantum-Crypto-Predictor/blob/main/FinalLongTermStrategy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# ------------------------------------------------------------------------------
# 🚀 FINAL STRATEGY: "THE PATIENT SNIPER" (5x Leverage / $250k Cap)
#    - Start: $100 + $50/mo
#    - Leverage: 5x (Safe, Institutional Grade)
#    - Filter: Only Trade above EMA 200
#    - Goal: Compound until $50k Balance (which equals $250k Position)
# ------------------------------------------------------------------------------
import os
import sys

# --- STEP 1: DEFINE SCRIPT ---
script_content = """
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import ta
import pytz
import yfinance as yf
import warnings
from datetime import timedelta

warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# --- CONFIGURATION ---
SYMBOL_YF = 'ETH-USD'
TIMEFRAME = '1h'       # Hourly Data
DATA_PERIOD = '2y'
CONFIDENCE_THRESHOLD = 0.60

# FINANCIAL SETTINGS
START_BALANCE = 100
MONTHLY_DEPOSIT = 50

# --- CRITICAL UPDATES ---
LEVERAGE = 5.0         # 5x Leverage (User Preference: Safety over Speed)
MAX_POSITION_SIZE = 250000 # GMX V2 Liquidity Ceiling
PROFIT_SPLIT = 0.50    # 50% Withdrawal Rate once Ceiling is hit

# 1:2 RISK REWARD RATIO
# At 5x Leverage, a 1.2% move = 6% Account impact (Very manageable)
TARGET_ROI = 0.024     # 2.4% Profit Target
STOP_LOSS = 0.012      # 1.2% Stop Loss
GMX_FEE = 0.001

# 1. FETCH DATA
print(f"\\n📡 FETCHING {DATA_PERIOD} OF ETH DATA (Hourly)...")
try:
    df = yf.download(tickers=SYMBOL_YF, period=DATA_PERIOD, interval=TIMEFRAME, progress=False)
    if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
    df.rename(columns={'Open': 'open', 'High': 'high', 'Low': 'low', 'Close': 'close', 'Volume': 'volume'}, inplace=True)
    df.index = pd.to_datetime(df.index, utc=True)
    df = df.tz_convert('America/Los_Angeles')

    # --- INDICATORS ---
    # 1. Trend Filter (EMA 200)
    df['EMA200'] = ta.trend.EMAIndicator(df['close'], window=200).ema_indicator()

    # 2. Quantum Features
    df['RSI'] = ta.momentum.RSIIndicator(df['close'], window=14).rsi()
    macd = ta.trend.MACD(df['close'])
    df['MACD_Diff'] = macd.macd_diff()
    df['Vol_Ratio'] = df['volume'] / df['volume'].rolling(20).mean()

    # Targets (Look forward 8 Hours)
    look_forward = 8
    targets = []
    for i in range(len(df) - look_forward):
        entry = df.iloc[i]['close']
        future = df.iloc[i+1 : i+1+look_forward]

        # Win if we hit +2.4% before -1.2%
        win_long = future['high'].max() >= (entry * (1 + TARGET_ROI))
        loss_long = future['low'].min() <= (entry * (1 - STOP_LOSS))

        win_short = future['low'].min() <= (entry * (1 - TARGET_ROI))
        loss_short = future['high'].max() >= (entry * (1 + STOP_LOSS))

        if win_long and not loss_long: targets.append(1)
        elif win_short and not loss_short: targets.append(0)
        else: targets.append(np.nan)

    df = df.iloc[:len(targets)]
    df['Target'] = targets

    # Filter: Only trade Active Hours + Volatility
    active_hours = [6, 7, 8, 9, 10, 16, 17, 18, 19, 20]
    df_clean = df[(df.index.dayofweek <= 4) & (df.index.hour.isin(active_hours))].dropna()

    # Normalize
    df_clean['n_RSI'] = df_clean['RSI'] / 100.0
    df_clean['n_MACD'] = (df_clean['MACD_Diff'] - df_clean['MACD_Diff'].min()) / (df_clean['MACD_Diff'].max() - df_clean['MACD_Diff'].min())
    df_clean['n_Vol'] = (df_clean['Vol_Ratio'] - df_clean['Vol_Ratio'].min()) / (df_clean['Vol_Ratio'].max() - df_clean['Vol_Ratio'].min())
    df_clean['n_Mom'] = np.where(df_clean['close'] > df_clean['open'], 0.8, 0.2)

    # Split
    start_date = df_clean.index[0]
    split_date = start_date + timedelta(days=180) # 6 Month Training
    train_df = df_clean[df_clean.index < split_date]
    test_df = df_clean[df_clean.index >= split_date]

except Exception as e:
    print(f"❌ Error: {e}")
    sys.exit(1)

# 2. TRAIN MODEL
qubits = [cirq.GridQubit(0, i) for i in range(4)]

def process_to_circuits(dataframe):
    circuits = []
    labels = []
    for _, row in dataframe.iterrows():
        c = cirq.Circuit()
        c.append(cirq.ry(row['n_RSI'] * np.pi)(qubits[0]))
        c.append(cirq.ry(row['n_MACD'] * np.pi)(qubits[1]))
        c.append(cirq.ry(row['n_Vol'] * np.pi)(qubits[2]))
        c.append(cirq.ry(row['n_Mom'] * np.pi)(qubits[3]))
        circuits.append(c)
        labels.append(row['Target'])
    return tfq.convert_to_tensor(circuits), np.array(labels)

print(f"\\n🧠 TRAINING MODEL (60% Precision)...")
X_train, y_train = process_to_circuits(train_df)
X_test, y_test = process_to_circuits(test_df)

params = sympy.symbols('theta0:14')
q_model_circuit = cirq.Circuit()
# (Same architecture)
q_model_circuit.append(cirq.rx(params[0])(qubits[0]))
q_model_circuit.append(cirq.ry(params[1])(qubits[1]))
q_model_circuit.append(cirq.rx(params[2])(qubits[2]))
q_model_circuit.append(cirq.ry(params[3])(qubits[3]))
q_model_circuit.append(cirq.CZ(qubits[0], qubits[1]))
q_model_circuit.append(cirq.CZ(qubits[2], qubits[3]))
q_model_circuit.append(cirq.ry(params[4])(qubits[0]))
q_model_circuit.append(cirq.rx(params[5])(qubits[1]))
q_model_circuit.append(cirq.ry(params[6])(qubits[2]))
q_model_circuit.append(cirq.rx(params[7])(qubits[3]))
q_model_circuit.append(cirq.CZ(qubits[1], qubits[2]))
q_model_circuit.append(cirq.rx(params[8])(qubits[0]))
q_model_circuit.append(cirq.ry(params[9])(qubits[1]))
q_model_circuit.append(cirq.rx(params[10])(qubits[2]))
q_model_circuit.append(cirq.rx(params[11])(qubits[3]))
q_model_circuit.append(cirq.ry(params[12])(qubits[0]))
q_model_circuit.append(cirq.rx(params[13])(qubits[0]))

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(), dtype=tf.string),
    tfq.layers.PQC(q_model_circuit, operators=cirq.Z(qubits[0])),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.005), loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=30, batch_size=32, verbose=0)

# 3. RUN SIMULATION
print(f"\\n💰 SIMULATION: 5x Leverage | $250k Max Cap | Patient Growth")
print("-" * 115)
print(f"{'Month':<6} | {'Date':<10} | {'Deps':<6} | {'Trades':<6} | {'Profit':<10} | {'Withdrawn':<10} | {'Balance'} | {'Pos Size'}")
print("-" * 115)

predictions = model.predict(X_test, verbose=0)
balance = START_BALANCE
last_deposit_date = test_df.index[0]
total_deposits = 0
total_withdrawn = 0
month_count = 1
monthly_profit = 0
trades_this_month = 0

for i in range(len(test_df)):
    row = test_df.iloc[i]
    current_date = row.name

    # --- END OF MONTH LOGIC ---
    if (current_date - last_deposit_date).days >= 30:

        raw_size = balance * LEVERAGE

        # Withdrawals (Only if we hit $250k Position size cap)
        # Note: At 5x leverage, this happens when Balance >= $50,000
        payout = 0
        if raw_size >= MAX_POSITION_SIZE and monthly_profit > 0:
            payout = monthly_profit * PROFIT_SPLIT
            balance -= payout
            total_withdrawn += payout

        # Add Deposit
        balance += MONTHLY_DEPOSIT
        total_deposits += MONTHLY_DEPOSIT

        display_size = min(balance * LEVERAGE, MAX_POSITION_SIZE)

        print(f"{month_count:<6} | {last_deposit_date.strftime('%Y-%m') :<10} | +$50   | {trades_this_month:<6} | ${monthly_profit:<9.2f} | ${payout:<9.2f} | ${balance:<9.2f} | ${display_size:,.0f}")

        last_deposit_date = current_date
        month_count += 1
        monthly_profit = 0
        trades_this_month = 0

    # --- TRADING LOGIC ---

    # 1. TREND FILTER: Only trade if Price > EMA 200
    if row['close'] < row['EMA200']:
        continue # Skip trade, market is weak

    # 2. QUANTUM CONFIDENCE
    prob = predictions[i][0]
    confidence = prob if prob > 0.5 else (1 - prob)
    if confidence < CONFIDENCE_THRESHOLD: continue

    # 3. EXECUTE
    raw_position = balance * LEVERAGE
    position_size = min(raw_position, MAX_POSITION_SIZE)

    if position_size < 10: continue

    fee = position_size * (GMX_FEE * 2)
    actual = row['Target']
    is_win = ((prob > 0.5) == actual)

    pnl = 0
    if is_win:
        pnl = (position_size * TARGET_ROI) - fee
    else:
        pnl = -((position_size * STOP_LOSS) + fee)

    balance += pnl
    monthly_profit += pnl
    trades_this_month += 1

    if balance <= 0: balance = 0

print("-" * 115)
print(f"🏁 FINAL STATS:")
print(f"   Total Deposited:  ${START_BALANCE + total_deposits}")
print(f"   Total Withdrawn:  ${total_withdrawn:,.2f} (Cash in Pocket)")
print(f"   Account Balance:  ${balance:,.2f} (Base Capital)")
print(f"   Net Value:        ${balance + total_withdrawn:,.2f}")
"""

with open("run_velocity_sim.py", "w") as f:
    f.write(script_content)

print("🚀 LAUNCHING 5x LEVERAGE SIMULATION...")
!/usr/local/envs/quantum_env/bin/python run_velocity_sim.py

🚀 LAUNCHING 5x LEVERAGE SIMULATION...
2025-12-02 22:56:15.664347: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-02 22:56:15.664401: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-02 22:56:15.666467: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-02 22:56:15.677823: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-02 22:5